# Multi-Disease Copy-Paste — Fresh Start: Paste + Align Only

This is a **clean restart**. No blending, no color/lighting matching yet — just:
1. Detect the leaf's orientation (angle of its long axis / vein direction)
2. Detect each lesion patch's orientation
3. Rotate the lesion to match the leaf's angle
4. Alpha-paste it directly onto the leaf (no Poisson/Gaussian/LAB matching)

Lighting, hue, saturation matching comes later once alignment looks right.

**Structure:**
- Cell Block 1 → BLB lesions pasted onto Brown Spot leaves (1 BLB per leaf, 15 random pairs)
- Cell Block 2 → Blast lesions pasted onto Brown Spot leaves (1 Blast per leaf, 15 random pairs)
- Cell Block 3 (bonus) → Healthy leaves get **both** 1 BLB + 1 Blast lesion, 15 random leaves


In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

import os, glob, random
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

random.seed(42)


Mounted at /content/drive


In [7]:
# ===== Config: EDIT THESE PATHS =====

# Base leaf folders (full leaf images, RGB, white/light background is fine)
BROWN_SPOT_DIR = "/content/drive/MyDrive/bacterial_leaf_blight"

# Lesion patch folders (each file can contain ONE or MULTIPLE lesion instances
# on a white/transparent background — we auto-split via connected components)

BLAST_PATCH_DIR = "/content/drive/MyDrive/blast_lesion_patches"

# Where to save composited previews (optional)
OUTPUT_DIR = "/content/drive/MyDrive/blast_on_blb"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_SAMPLES = 10**9   # how many random leaf/patch pairs to try per class

IMG_EXTS = ("*.png", "*.jpg", "*.jpeg")

def list_images(folder):
    files = []
    for ext in IMG_EXTS:
        files.extend(glob.glob(os.path.join(folder, ext)))
    return sorted(files)


In [8]:
# ===== Core helpers: leaf mask, angle detection, lesion splitting, rotation, pasting =====

def get_leaf_mask(leaf_rgb):
    """Segment the leaf from a white/light background using Otsu on saturation,
    then keep only the largest connected component."""
    hsv = cv2.cvtColor(leaf_rgb, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1]
    _, mask = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = np.where(labels == largest, 255, 0).astype(np.uint8)
    return mask


def get_principal_angle(mask):
    """PCA long-axis angle (degrees) of a binary mask. 180-degree ambiguous,
    which is fine for aligning elongated streak-shaped lesions."""
    ys, xs = np.where(mask > 0)
    pts = np.column_stack([xs, ys]).astype(np.float64)
    mean = pts.mean(axis=0)
    centered = pts - mean
    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    principal = eigvecs[:, np.argmax(eigvals)]
    angle = np.degrees(np.arctan2(principal[1], principal[0]))
    return angle, mean


def load_rgba(path):
    return np.array(Image.open(path).convert("RGBA"))


def extract_lesion_instances(patch_rgba, min_area=150):
    """Split a patch file into individual lesion instances via connected
    components (handles files that contain multiple scattered lesion crops)."""
    if patch_rgba[:, :, 3].min() < 250:
        mask = (patch_rgba[:, :, 3] > 20).astype(np.uint8) * 255
    else:
        rgb = patch_rgba[:, :, :3]
        white = np.all(rgb > 240, axis=2)
        mask = (~white).astype(np.uint8) * 255

    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    instances = []
    for i in range(1, num):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < min_area:
            continue
        x, y, w, h, _ = stats[i]
        comp_mask = np.zeros_like(mask)
        comp_mask[labels == i] = 255
        crop_mask = comp_mask[y:y + h, x:x + w]
        crop_rgb = patch_rgba[y:y + h, x:x + w, :3]
        crop_rgba = np.dstack([crop_rgb, crop_mask])
        angle, _ = get_principal_angle(crop_mask)
        instances.append({"rgba": crop_rgba, "angle": angle, "area": area})
    return instances


def rotate_rgba(patch_rgba, angle_deg):
    """Rotate an RGBA lesion patch about its own center, then tight-crop."""
    h, w = patch_rgba.shape[:2]
    diag = int(np.ceil(np.hypot(h, w))) + 4
    canvas = np.zeros((diag, diag, 4), dtype=np.uint8)
    y0, x0 = (diag - h) // 2, (diag - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = patch_rgba
    center = (diag / 2, diag / 2)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    rotated = cv2.warpAffine(
        canvas, M, (diag, diag),
        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0)
    )
    alpha = rotated[:, :, 3]
    ys, xs = np.where(alpha > 10)
    if len(xs) == 0:
        return rotated
    y1, y2, x1, x2 = ys.min(), ys.max(), xs.min(), xs.max()
    return rotated[y1:y2 + 1, x1:x2 + 1]


def find_paste_position(leaf_mask, lesion_alpha, occupied_mask=None, max_tries=300, min_coverage=0.85):
    """Find a top-left (x0, y0) so the lesion sits fully inside the leaf mask
    and (optionally) doesn't overlap an already-placed lesion."""
    lh, lw = lesion_alpha.shape
    H, W = leaf_mask.shape
    ys, xs = np.where(leaf_mask > 0)
    if len(xs) == 0:
        return None
    lesion_bin = lesion_alpha > 10
    for _ in range(max_tries):
        idx = random.randrange(len(xs))
        cx, cy = xs[idx], ys[idx]
        x0, y0 = cx - lw // 2, cy - lh // 2
        x1, y1 = x0 + lw, y0 + lh
        if x0 < 0 or y0 < 0 or x1 > W or y1 > H:
            continue
        region = leaf_mask[y0:y1, x0:x1] > 0
        coverage = region[lesion_bin].sum() / max(lesion_bin.sum(), 1)
        if coverage < min_coverage:
            continue
        if occupied_mask is not None:
            occ_region = occupied_mask[y0:y1, x0:x1] > 0
            if occ_region[lesion_bin].any():
                continue
        return x0, y0
    return None


def paste_lesion(leaf_rgb, lesion_rgba, x0, y0):
    """Simple alpha paste. No color/lighting adjustment."""
    out = leaf_rgb.copy()
    lh, lw = lesion_rgba.shape[:2]
    alpha = lesion_rgba[:, :, 3:4].astype(np.float32) / 255.0
    region = out[y0:y0 + lh, x0:x0 + lw, :3].astype(np.float32)
    lesion_rgb = lesion_rgba[:, :, :3].astype(np.float32)
    blended = region * (1 - alpha) + lesion_rgb * alpha
    out[y0:y0 + lh, x0:x0 + lw, :3] = blended.astype(np.uint8)
    return out


In [9]:
# ===== Composite driver =====

def paste_one_disease_on_leaf(leaf_rgb, leaf_mask, leaf_angle, patch_files, occupied_mask=None):
    """Pick a random patch file -> random lesion instance inside it -> rotate to
    match leaf angle -> paste at a valid random spot on the leaf.
    Returns (result_rgb, updated_occupied_mask, used_patch_path) or (None, occupied_mask, None)."""
    candidates = patch_files[:]
    random.shuffle(candidates)
    for p in candidates:
        patch_rgba = load_rgba(p)
        instances = extract_lesion_instances(patch_rgba)
        if not instances:
            continue
        random.shuffle(instances)
        for inst in instances:
            angle_diff = leaf_angle - inst["angle"]
            rotated = rotate_rgba(inst["rgba"], -angle_diff)
            pos = find_paste_position(leaf_mask, rotated[:, :, 3], occupied_mask=occupied_mask)
            if pos is None:
                continue
            x0, y0 = pos
            result = paste_lesion(leaf_rgb, rotated, x0, y0)
            if occupied_mask is None:
                occupied_mask = np.zeros(leaf_mask.shape, dtype=np.uint8)
            new_occ = occupied_mask.copy()
            lh, lw = rotated.shape[:2]
            lesion_bin = (rotated[:, :, 3] > 10).astype(np.uint8) * 255
            new_occ[y0:y0 + lh, x0:x0 + lw] = np.maximum(new_occ[y0:y0 + lh, x0:x0 + lw], lesion_bin)
            return result, new_occ, p
    return None, occupied_mask, None


def show_grid(images, titles, cols=5, figsize_per=(3, 3.4)):
    n = len(images)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per[0]*cols, figsize_per[1]*rows))
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(images[i])
            ax.set_title(titles[i], fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## Cell Block 2 — Blast lesions on Brown Spot leaves (1 Blast per leaf)

In [ ]:
# ===== Cell Block 2: Blast -> Brown Spot =====
brown_spot_files = list_images(BROWN_SPOT_DIR)
blast_patch_files = list_images(BLAST_PATCH_DIR)
assert len(blast_patch_files) > 0, "No Blast patches found — check BLAST_PATCH_DIR"

leaf_sample2 = random.sample(brown_spot_files, min(N_SAMPLES, len(brown_spot_files)))

results_blast, titles_blast = [], []
for leaf_path in leaf_sample2:
    leaf_rgb = np.array(Image.open(leaf_path).convert("RGB"))
    leaf_mask = get_leaf_mask(leaf_rgb)
    leaf_angle, _ = get_principal_angle(leaf_mask)

    result, occ, used_patch = paste_one_disease_on_leaf(leaf_rgb, leaf_mask, leaf_angle, blast_patch_files)

    if result is None:
        print(f"[skip] no valid placement for {os.path.basename(leaf_path)}")
        continue

    results_blast.append(result)
    titles_blast.append(os.path.basename(leaf_path))

    out_path = os.path.join(OUTPUT_DIR, f"blast_on_brownspot_{os.path.basename(leaf_path)}")
    Image.fromarray(result).save(out_path)

print(f"Composited {len(results_blast)}/{len(leaf_sample2)} Blast-on-BrownSpot images")
show_grid(results_blast, titles_blast, cols=5)


Composited 1075/1075 Blast-on-BrownSpot images


## Cell Block 3 (bonus) — Healthy leaves get 1 BLB + 1 Blast each

In [ ]:
# ===== Cell Block 3: BLB + Blast -> Healthy =====

healthy_files = list_images(HEALTHY_DIR)
assert len(healthy_files) > 0, "No healthy leaves found — check HEALTHY_DIR"

leaf_sample3 = random.sample(healthy_files, min(N_SAMPLES, len(healthy_files)))

results_healthy, titles_healthy = [], []
for leaf_path in leaf_sample3:
    leaf_rgb = np.array(Image.open(leaf_path).convert("RGB"))
    leaf_mask = get_leaf_mask(leaf_rgb)
    leaf_angle, _ = get_principal_angle(leaf_mask)

    # first lesion: BLB
    result1, occ1, used1 = paste_one_disease_on_leaf(leaf_rgb, leaf_mask, leaf_angle, blb_patch_files)
    if result1 is None:
        print(f"[skip] no valid BLB placement for {os.path.basename(leaf_path)}")
        continue

    # second lesion: Blast, avoiding the area already occupied by the BLB lesion
    result2, occ2, used2 = paste_one_disease_on_leaf(result1, leaf_mask, leaf_angle, blast_patch_files, occupied_mask=occ1)
    if result2 is None:
        print(f"[skip] no valid Blast placement for {os.path.basename(leaf_path)}")
        continue

    results_healthy.append(result2)
    titles_healthy.append(os.path.basename(leaf_path))

    out_path = os.path.join(OUTPUT_DIR, f"blb_blast_on_healthy_{os.path.basename(leaf_path)}")
    Image.fromarray(result2).save(out_path)

print(f"Composited {len(results_healthy)}/{len(leaf_sample3)} BLB+Blast-on-Healthy images")
show_grid(results_healthy, titles_healthy, cols=5)
